# Template, with $LINK as example

### Download data

In [10]:
import yfinance as yf
import numpy as np
import pandas as pd

ticker = yf.Ticker("LINK")

data = ticker.history(start="2021-01-01", auto_adjust=True)

price = data["Close"].dropna()
returns = np.log(price / price.shift(1)).dropna()

print(price.tail())


Failed to get ticker 'LINK' reason: Expecting value: line 1 column 1 (char 0)
$LINK: possibly delisted; No timezone found


Series([], Name: Close, dtype: float64)


### Set up GARCH

In [11]:
from arch import arch_model

model = arch_model(returns*100, vol='Garch', p=1, q=1, dist='t')
res = model.fit(disp='off')
print(res.summary())


ValueError: first_obs and last_obs produce an empty array.

__Comment:__ we use GARCH because without options data, GARCH gives more stable volatility estimates than Heston.

### MONTE CARLO Simulations

In [ ]:
sim_days = {
    "1M":21,
    "3M":63,
    "1Y":252,
    "3Y":756,
    "5Y":1260
}

n_sims = 10000
S0 = price.iloc[-1]

def simulate(days):
    sim = res.forecast(horizon=days, method='simulation', simulations=n_sims)
    vols = np.sqrt(sim.variance.values[-1])
    shocks = np.random.standard_t(df=8, size=(n_sims,days))
    paths = S0*np.exp(np.cumsum(vols*shocks/100, axis=1))
    return paths[:,-1]

results = {k:simulate(v) for k,v in sim_days.items()}


### Print targets

In [ ]:
for k,v in results.items():
    print(k,
          "P10:",np.percentile(v,10),
          "P50:",np.percentile(v,50),
          "P90:",np.percentile(v,90))
